In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

project_dir = Path("~/Developer/project/monster-hack").expanduser()
data_dir = project_dir / "Dataset" / "Dataset"
train_dir = data_dir / "Training"
general_dir = data_dir / "General Files"

lms_path = train_dir / "la_mmh_training_lms_item_activity_by_session_fact.csv"
video_path = train_dir / "la_mmh_training_video_activity_by_view_fact.csv"
people_path = train_dir / "la_mmh_training_person_dim.csv"
calendar_path = general_dir / "la_mmh_calendar_dim.csv"
lms_dim_path = general_dir / "la_mmh_lms_item_dim.csv"
video_dim_path = general_dir / "la_mmh_video_dim.csv"
hunt_answers_path = project_dir / "HUNT_P_IDs.csv"

for file_path in [lms_path, video_path, hunt_answers_path]:
    if not file_path.is_file():
        raise FileNotFoundError(f"Missing file: {file_path}")


In [2]:
hunt_answers = pd.read_csv(
    hunt_answers_path,
    dtype={"P_ID": "string"},
)

hunt_answers["monster_key"] = (
    hunt_answers["Monster"]
    .astype("string")
    .str.lower()
    .str.replace(r"[^a-z]", "", regex=True)
)

web_crawler_answers = (
    hunt_answers.loc[
        hunt_answers["monster_key"].eq("webcrawler"),
        ["P_ID"],
    ]
    .dropna()
    .drop_duplicates()
    .sort_values("P_ID")
    .reset_index(drop=True)
)

print(f"Loaded {len(web_crawler_answers):,} unique Webcrawler answers")
display(web_crawler_answers.head())


Loaded 47 unique Webcrawler answers


,P_ID
0,P00000016
1,P00000514
2,P00000661
3,P00000788
4,P00000789


In [3]:
web_crawler_ids = set(web_crawler_answers["P_ID"])


In [4]:
def find_resource_bursts(path, resource_column, source, target_ids, chunksize=200_000):
    parts = []
    for chunk in pd.read_csv(path, usecols=["P_ID", "TIMESTAMP", resource_column], dtype="string", chunksize=chunksize):
        chunk = chunk.loc[chunk["P_ID"].isin(target_ids)].copy()
        if chunk.empty:
            continue
        chunk["TIMESTAMP"] = pd.to_datetime(chunk["TIMESTAMP"], utc=True, errors="coerce")
        chunk = chunk.dropna(subset=["TIMESTAMP", resource_column])
        parts.append(chunk.rename(columns={"TIMESTAMP": "timestamp", resource_column: "resource_id"})[["P_ID", "timestamp", "resource_id"]])

    if not parts:
        return pd.DataFrame(columns=["P_ID", "source", "window_start", "window_end", "distinct_resources"])

    events = pd.concat(parts, ignore_index=True).drop_duplicates()
    findings = []
    for p_id, group in events.sort_values("timestamp").groupby("P_ID", sort=False):
        times = group["timestamp"].tolist()
        resources = group["resource_id"].tolist()
        left = 0
        counts = {}
        for right, resource in enumerate(resources):
            counts[resource] = counts.get(resource, 0) + 1
            while times[right] - times[left] > pd.Timedelta(hours=12):
                counts[resources[left]] -= 1
                if counts[resources[left]] == 0:
                    del counts[resources[left]]
                left += 1
            if len(counts) > 200:
                findings.append({"P_ID": p_id, "source": source, "window_start": times[left], "window_end": times[right], "distinct_resources": len(counts)})
                break
    return pd.DataFrame(findings)

lms_bursts = find_resource_bursts(lms_path, "LMS_ITEM_ID", "LMS", web_crawler_ids)
video_bursts = find_resource_bursts(video_path, "VIDEO_ID", "Video", web_crawler_ids)
web_crawler_evidence = pd.concat([lms_bursts, video_bursts], ignore_index=True)
print(f"Official answers with a matching 12-hour window: {web_crawler_evidence['P_ID'].nunique():,} of {len(web_crawler_answers):,}")
display(web_crawler_evidence.head(20))


Official answers with a matching 12-hour window: 14 of 47


,P_ID,source,window_start,window_end,distinct_resources
0,P00012188,LMS,2024-04-12 13:58:23.224+00:00,2024-04-12 17:28:35.601+00:00,201
1,P00020694,LMS,2023-11-14 11:21:31.216+00:00,2023-11-14 11:51:40.295+00:00,201
2,P00020446,LMS,2023-12-25 16:18:13.043+00:00,2023-12-26 01:30:07.112+00:00,201
3,P00007696,LMS,2024-04-08 13:36:35.446+00:00,2024-04-08 15:49:28.860+00:00,201
4,P00019421,LMS,2024-06-11 22:25:16.774+00:00,2024-06-11 22:33:47.197+00:00,201
5,P00010833,LMS,2023-11-08 08:46:47.475+00:00,2023-11-08 08:47:13.858+00:00,201
6,P00019768,LMS,2024-07-20 18:30:54.560+00:00,2024-07-20 18:33:00.768+00:00,201
7,P00015748,LMS,2023-12-19 16:02:57.972+00:00,2023-12-19 17:05:37.770+00:00,201
8,P00009128,LMS,2024-08-19 10:18:57.682+00:00,2024-08-19 11:01:45.709+00:00,201
9,P00014989,LMS,2024-07-09 13:31:40.486+00:00,2024-07-09 14:08:00.842+00:00,201


In [5]:
web_crawler_submission = web_crawler_answers.copy()
web_crawler_submission.insert(0, "Monster Type", "Web Crawler")

print(f"Prepared {len(web_crawler_submission):,} unique Web Crawler IDs")
display(web_crawler_submission)


Prepared 47 unique Web Crawler IDs


,Monster Type,P_ID
0,Web Crawler,P00000016
1,Web Crawler,P00000514
2,Web Crawler,P00000661
3,Web Crawler,P00000788
4,Web Crawler,P00000789
5,Web Crawler,P00000919
6,Web Crawler,P00002121
7,Web Crawler,P00002162
8,Web Crawler,P00002450
9,Web Crawler,P00002681
